# 1. Configuración Inicial y Conexión a Base de Datos
Definimos las rutas de origen de los CSV y la cadena de conexión a SQL Server.

In [1]:
import os
import re
import pyodbc
from pathlib import Path

ORIGEN = Path.cwd() / 'Descargas' / 'Oxford'
TABLA_STAGE = 'dbo.avance_oxford_stage_new'
CONN_STR = 'DRIVER={ODBC Driver 17 for SQL Server};SERVER=CCA-CENT-R5186;DATABASE=rendimiento_academico;Trusted_Connection=yes;'

# 2. Funciones de Limpieza y Extracción de Datos
Expresiones regulares para extraer clase y grupo del nombre del archivo, y transformaciones de fechas y porcentajes.

In [2]:
PATRON_CURSO = re.compile(r"""(?ix)
    (?P<clase>[a-z]{1,3}\s*\d+(?:[.\-_]\d+)?)    # B1.2, B1-2, A2, C1.1
    [\s\-_]*
    (?P<grupo>\d{1,2}\s*[ap]\.?m\.?)             # 5PM, 10AM
    [\s\-_]*
    (?P<codigo>\d+)                              # 28
    (?:[\s\-_]+(?P<segmento>[a-z]{2,10}))?       # IAV, TIC, BPO, GT
""")

MESES = {"Jan": "01", "Feb": "02", "Mar": "03", "Apr": "04", "May": "05", "Jun": "06",
         "Jul": "07", "Aug": "08", "Sep": "09", "Oct": "10", "Nov": "11", "Dec": "12"}

def partes_del_nombre(nombre_archivo):
    """Extrae clase, código, grupo y segmento del nombre del archivo."""
    nombre = Path(nombre_archivo).stem
    nombre = re.sub(r"^\d{4}-\d{2}-\d{2}_\d{3,4}_", "", nombre)
    hallado = PATRON_CURSO.search(nombre)
    if not hallado:
        return {"clase": None, "codigo": None, "grupo": None, "segmento": None}
    
    return {
        "clase": re.sub(r"[\s_\-]", ".", hallado["clase"].strip()).upper(),
        "codigo": hallado["codigo"],
        "grupo": re.sub(r"[\s.]", "", hallado["grupo"]).upper(),
        "segmento": (hallado["segmento"] or "").upper() or None,
    }

def a_porcentaje(val):
    """Convierte '33.3%' -> 33.3 | '-', '', 'Never' -> None"""
    val = str(val).strip(' \"')
    if val in ("", "-", "Never"):
        return None
    try:
        return float(val.replace("%", ""))
    except ValueError:
        return None

def a_fecha(val):
    """Convierte '30 Sep 2026' -> '2026-09-30'"""
    val = str(val).strip(' \"')
    if val in ("", "-", "Never"):
        return None
    try:
        dia, mes, anio = val.split()
        return f"{anio}-{MESES[mes]}-{dia.zfill(2)}"
    except Exception:
        return None

# 3. Validación de Estructura de Columnas
Nos aseguramos de que el reporte de Oxford mantenga las mismas ubicaciones de las unidades.

In [3]:
UNIT_CATEGORIES = {
    1: "achievements", 2: "news", 3: "frontiers", 4: "processes",
    5: "survival", 6: "trends", 7: "surroundings", 8: "change",
    9: "reviews", 10: "mysteries", 11: "wishes", 12: "wisdom",
}

def validar_estructura(row1, nombre_archivo):
    """Verifica que las columnas caigan exactamente donde esperamos y con la categoría correcta."""
    if len(row1) < 54:
        raise ValueError(f"El archivo tiene {len(row1)} columnas, se esperaban al menos 54.")
    
    if not row1[4].strip(' \"').startswith("Total"):
        raise ValueError("La columna índice 4 no es 'Total'.")
        
    for n in range(1, 13):
        base = 6 + (n - 1) * 4
        val_base = row1[base].strip(' \"')
        val_test = row1[base+2].strip(' \"')
        
        # Construimos el título esperado usando la categoría de la unidad
        categoria = UNIT_CATEGORIES[n]
        esperado_base = f"Unit {n} {categoria}"
        
        # Normalizamos espacios (eliminar espacios extra) y pasamos a minúsculas para comparar exactitud
        val_base_norm = " ".join(val_base.lower().split())
        
        if val_base_norm != esperado_base.lower():
            raise ValueError(f"Error en archivo '{nombre_archivo}', Unidad {n}: Se esperaba la columna '{esperado_base}' pero se encontró '{val_base}'. Verifica que el archivo corresponda al nivel esperado.")
            
        if not val_test.lower().startswith(f"unit {n} test"):
            raise ValueError(f"Error en archivo '{nombre_archivo}', Unidad {n}: Se esperaba una columna que comience con 'Unit {n} Test' pero se encontró '{val_test}'.")


SyntaxError: unexpected character after line continuation character (3777667781.py, line 1)

# 4. Proceso de Carga y Ejecución en SQL
Se lee cada archivo `.csv`, se transforma a formato ancho, y se envía a SQL en bloque por cada archivo. Al finalizar con éxito, se ejecuta el procedimiento almacenado.

In [43]:
def procesar_archivos_oxford(origen_dir):
    conn = pyodbc.connect(CONN_STR)
    cursor = conn.cursor()
    cursor.fast_executemany = True  # Inserción optimizada por lotes
    
    cursor.execute(f"TRUNCATE TABLE {TABLA_STAGE}")
    conn.commit()
    print(f"[{TABLA_STAGE}] limpia y lista para cargar.")
    
    archivos = list(Path(origen_dir).glob("*.csv"))
    total_procesados = 0
    total_cargados = 0
    errores = []
    
    # Construcción dinámica de las columnas de las unidades usando UNIT_CATEGORIES
    columnas_unidades = []
    for n in range(1, 13):
        cat = UNIT_CATEGORIES[n]
        u = f"u{n:02d}"
        # Genera ej: u01_achievements_completed_pct, u01_achievements_score_pct, u01_test_completed_pct...
        columnas_unidades.append(f"{u}_{cat}_completed_pct, {u}_{cat}_score_pct, {u}_test_completed_pct, {u}_test_score_pct")
    
    cols_unidades_str = ",\n        ".join(columnas_unidades)
    placeholders_unidades = ",\n        ".join(["?, ?, ?, ?"] * 12)
    
    sql_insert = f\"\"\"
    INSERT INTO {TABLA_STAGE} (
        teacher, learning_material, student, last_opened, clase, codigo, grupo, segmento,
        total_completed_pct, total_score_pct,
        {cols_unidades_str}
    ) VALUES (
        ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 
        {placeholders_unidades}
    )
    \"\"\"
    
    for archivo in archivos:
        try:
            with open(archivo, "r", encoding="utf-16", errors="replace") as f:
                lineas = [l.rstrip('\r\n') for l in f if l.strip()]
                
                if len(lineas) < 4:
                    raise ValueError("El archivo no tiene las 4 líneas mínimas requeridas.")
                
                rows = [line.split('\t') for line in lineas]
                
                # Se envía también el nombre del archivo para enriquecer el error
                validar_estructura(rows[0], archivo.name)
                
                row_class_avg = rows[2]
                if "Class Average" not in row_class_avg[2].strip(' \"'):
                    raise ValueError(f"La fila 3 no es 'Class Average'. Estudiante: '{row_class_avg[2]}'")
                
                teacher_val = row_class_avg[0].strip(' \"') if len(row_class_avg) > 0 else None
                material_val = row_class_avg[1].strip(' \"') if len(row_class_avg) > 1 else None
                
                meta = partes_del_nombre(archivo.name)
                params_lote = []
                
                for row in rows[3:]:
                    if not any(c.strip() for c in row): 
                        continue
                    
                    if len(row) < 54:
                        row.extend([""] * (54 - len(row)))
                    
                    student = row[2].strip(' \"')
                    last_opened = a_fecha(row[3])
                    total_comp = a_porcentaje(row[4])
                    total_score = a_porcentaje(row[5])
                    
                    param_row = [
                        teacher_val, material_val, student, last_opened,
                        meta["clase"], meta["codigo"], meta["grupo"], meta["segmento"],
                        total_comp, total_score
                    ]
                    
                    for n in range(1, 13):
                        base = 6 + (n - 1) * 4
                        param_row.append(a_porcentaje(row[base]))
                        param_row.append(a_porcentaje(row[base+1]))
                        param_row.append(a_porcentaje(row[base+2]))
                        param_row.append(a_porcentaje(row[base+3]))
                    
                    params_lote.append(param_row)
                
                cursor.executemany(sql_insert, params_lote)
                conn.commit()
                
                total_procesados += 1
                total_cargados += len(params_lote)
                
        except Exception as e:
            errores.append(f"{archivo.name} -> {e}")
            
    if errores:
        print("\n--- ERRORES ENCONTRADOS ---")
        for e in errores:
            print(f"- {e}")
        print("\n[!] Hubo errores. Los archivos correctos subieron al stage, pero NO se ejecutó el procedimiento almacenado.")
    else:
        print(f"\nTodos los {total_procesados} archivos fueron cargados correctamente.")
        print("Ejecutando sp_procesar_corte_avance_oxford...")
        cursor.execute("EXEC dbo.sp_procesar_corte_avance_oxford")
        conn.commit()
        print("Procedimiento ejecutado con éxito.")
        
    print(f"\nResumen: {total_procesados} archivos procesados, {total_cargados} estudiantes cargados al stage.")
    cursor.close()
    conn.close()

# Iniciar el proceso
procesar_archivos_oxford(ORIGEN)


[dbo.avance_oxford_stage_new] limpia y lista para cargar.

Todos los 1 archivos fueron cargados correctamente.
Ejecutando sp_procesar_corte_avance_oxford...
Procedimiento ejecutado con éxito.

Resumen: 1 archivos procesados, 1 estudiantes cargados al stage.
